# Time Series Analysis: iSAX

In [76]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import norm

## 1. iSAX Representation

In [77]:
class Encoder():
    def __init__(self, word_length, max_cardinality=256):
        self.n_segments = word_length
        self.max_cardinality = max_cardinality
        
        # Pre-compute the boundaries on initialization
        self.breakpoint_table = self._build_breakpoint_table()

    def _build_breakpoint_table(self):
        table = {}
        c = 2
        
        # iSAX cardinalities iteratively double (2, 4, 8, 16, 32...)
        while c <= self.max_cardinality:
            # We need c-1 breakpoints to divide the space into c bins
            percentiles = np.arange(1, c) / c
            
            # norm.ppf finds the exact Z-score boundaries for those percentiles
            table[c] = norm.ppf(percentiles)
            
            c *= 2
            
        return table
    
    def get_bounds(self, binary_symbol, cardinality):
        # Fetch the pre-computed array of boundaries for this cardinality
        breakpoints = self.breakpoint_table[cardinality]
        
        # Convert the binary string (e.g., '10') back to a base-10 integer index
        symbol_index = int(binary_symbol, 2)
        
        # Scenario 1: The bottom bin (extends to negative infinity)
        if symbol_index == 0:
            beta_L = -np.inf
            beta_U = breakpoints[0]
            
        # Scenario 2: The top bin (extends to positive infinity)
        elif symbol_index == cardinality - 1:
            beta_L = breakpoints[-1]
            beta_U = np.inf
            
        # Scenario 3: Caught between two boundaries
        else:
            beta_L = breakpoints[symbol_index - 1]
            beta_U = breakpoints[symbol_index]
            
        return beta_L, beta_U

    def paa(self, ts):
        # split_segments creates a list of sub-arrays. 
        split_segments = np.array_split(ts, self.n_segments)

        # Calculate the mean for each segment and wrap it back into a numpy array
        ts_paa = np.array([np.mean(seg) for seg in split_segments])
        
        return ts_paa
    
    def iSAX(self, ts, cardinalities):
        # 1. Z-Normalize the time series
        ts_normalized = (ts - np.mean(ts)) / (np.std(ts) + 1e-8)
        
        # 2. Compute PAA using the class method
        ts_paa = self.paa(ts_normalized)

        isax_symbols = []
        
        # 3. Convert PAA values to binary symbols using the pre-computed table
        for i in range(self.n_segments):
            c = cardinalities[i]
            breakpoints = self.breakpoint_table[c] 
            
            paa_value = ts_paa[i]
            
            # np.searchsorted efficiently finds which bin the value falls into
            symbol_index = np.searchsorted(breakpoints, paa_value)
            
            # Convert the integer index to a zero-padded binary string
            num_bits = int(np.log2(c))
            binary_symbol = format(symbol_index, f'0{num_bits}b')
            
            isax_symbols.append(binary_symbol)

        # Return as a tuple so it can be hashed as a dictionary key in the Index
        return tuple(isax_symbols)

In [ ]:
def lower_cardinality(query_isax, steps):
    # Safety check: If steps is 0, just return the original word!
    if steps == 0:
        return query_isax
        
    return tuple(symbol[:-steps] for symbol in query_isax)

def promote_cardinality(query_isax, goal_isax):
    promoted_isax = []
    
    for q_sym, g_sym in zip(query_isax, goal_isax):
        
        # Calculate how many bits we need to add
        diff = len(g_sym) - len(q_sym)
        
        if diff <= 0:
            promoted_isax.append(q_sym)
            continue
            
        # Isolate the prefix of the goal symbol that matches our current length
        prefix = g_sym[:len(q_sym)]
        
        # Rule 1: If it's a perfect prefix, the unknown bits match the goal bits
        if q_sym == prefix:
            promoted_isax.append(g_sym)
            
        # Rule 2: If it's lexicographically smaller, all unknown bits become 1
        elif q_sym < prefix:
            promoted_isax.append(q_sym + ('1' * diff))
            
        # Rule 3: If it's lexicographically larger, all unknown bits become 0
        else:
            promoted_isax.append(q_sym + ('0' * diff))
            
    return tuple(promoted_isax)

## 2. iSAX Index

In [78]:
import heapq

class iSaxNode():
    def __init__(self, isax_word, cardinalities, threshold, word_length, split_dim, encoder):
        # Constant parameters across nodes
        self.threshold = threshold
        self.word_length = word_length
        self.encoder = encoder

        # ISax of current node
        self.isax_word = isax_word
        self.cardinalities = cardinalities
        self.split_dim = split_dim

        # Traversal variables
        self.children_hash = {}
        self.file = []
        self.is_terminal = True # Node is always leaf when created

    @property
    def children(self):
        return list(self.children_hash.values())

    def is_full(self):
        return len(self.file) >= self.threshold    

    def split_node(self):
        # 1: Calculate the elevated cardinalities for the children
        new_cardinalities = self.cardinalities.copy()
        new_cardinalities[self.split_dim] *= 2
        
        next_split_dim = (self.split_dim + 1) % self.word_length

        # 2: Redistribute existing time series to children
        for ts in self.file:
            ts_isax = self.encoder.iSAX(ts, cardinalities=new_cardinalities)
            
            if ts_isax not in self.children_hash:
                self.children_hash[ts_isax] = iSaxNode(
                    ts_isax, new_cardinalities, self.threshold, self.word_length, next_split_dim, self.encoder
                )
            self.children_hash[ts_isax].insert(ts)
            
        # 3: Update node to be internal
        self.is_terminal = False
        self.file = []

    def insert(self, ts):
        # 1: Determine node type
        if self.is_terminal:
            # 2.1: Items are always added to leafs
            self.file.append(ts)

            # 2.2: Split leaf if it overflows
            if self.is_full():
                self.split_node()
        else:
            # 3.1: Get time series isax for children when internal node
            target_cardinalities = self.cardinalities.copy()
            target_cardinalities[self.split_dim] *= 2
            
            ts_isax = self.encoder.iSAX(ts, cardinalities=target_cardinalities)

            # 3.2: Children are created lazily
            if ts_isax not in self.children_hash:
                next_split_dim = (self.split_dim + 1) % self.word_length
                self.children_hash[ts_isax] = iSaxNode(
                    ts_isax, target_cardinalities, self.threshold, self.word_length, next_split_dim, self.encoder
                )

            # 3.3: Route to child with respective isax
            self.children_hash[ts_isax].insert(ts)

    def calc_min_dist(self, query):
        best_ts = None
        best_dist = np.inf

        # Perform serial scan across node content
        for ts in self.file:
            # Euclidean distance
            dist = np.sqrt(np.sum((ts - query) ** 2))
            if dist < best_dist:
                best_dist = dist
                best_ts = ts

        return best_ts, best_dist
    
    def route_to_child(self, query):
        # 1. Predict the iSAX word using the elevated cardinalities
        target_cardinalities = self.cardinalities.copy()
        target_cardinalities[self.split_dim] *= 2
        query_iSAX = self.encoder.iSAX(query, target_cardinalities)

        # 2. Ideal Scenario: The exact hash path exists
        if query_iSAX in self.children_hash:
            return self.children_hash[query_iSAX]
        
        # 3. Fallback Heuristic 1: Match on the split dimension
        query_split_symbol = query_iSAX[self.split_dim]
        
        for child_word, child_node in self.children_hash.items():
            if child_word[self.split_dim] == query_split_symbol:
                return child_node
                
        # 4. Fallback Heuristic 2: Absolute failure, pick the first available child
        return list(self.children_hash.values())[0]
        
    def mindist_paa_isax(self, query):
        # 1: Calculate PAA of query time series
        query_normalized = (query - np.mean(query)) / (np.std(query) + 1e-8)
        query_paa = self.encoder.paa(query_normalized)

        # 2: Loop through every aggregated position
        dist_sq = 0.0
        for i in range(self.word_length):
            # 2.1: Fetch boundaries of the node's isax word
            symbol = self.isax_word[i]
            c = self.cardinalities[i]

            breakpoint_L, breakpoint_U = self.encoder.get_bounds(symbol, c)

            # 2.2: Calculate distance of paa to boundaries
            paa_val = query_paa[i]

            if breakpoint_L > paa_val:
                dist_sq += (breakpoint_L - paa_val) ** 2
            elif breakpoint_U < paa_val:
                dist_sq += (breakpoint_U - paa_val) ** 2

        # 3: Return normalized root of distance
        return np.sqrt((len(query) / self.word_length) * dist_sq)

In [79]:
class iSaxIndex():
    def __init__(self, word_length=8, threshold=100, base_cardinality=4, max_cardinality=256):
        self.word_length = word_length
        self.threshold = threshold
        self.base_cardinalities = [base_cardinality] * word_length

        self.encoder = Encoder(word_length, max_cardinality)
        
        # Root node is stored as simple hash (only routes to children)
        self.root_hash = {}
        
    def insert(self, ts):
        # Calculate the base word
        base_word = self.encoder.iSAX(ts, self.base_cardinalities)
        
        # If this base word has never been seen, create a new terminal leaf for it
        if base_word not in self.root_hash:
            self.root_hash[base_word] = iSaxNode(
                isax_word=base_word,
                cardinalities=self.base_cardinalities,
                threshold=self.threshold, 
                word_length=self.word_length, 
                split_dim=0,
                encoder=self.encoder
            )
            
        # Push the data into the appropriate base-level node
        self.root_hash[base_word].insert(ts)
    
    def approximate_search(self, query):
        # 1: Start at the root dictionary
        base_word = self.encoder.iSAX(query, self.base_cardinalities)
        
        if base_word in self.root_hash:
            node = self.root_hash[base_word]
        else:
            # Fallback at the Root Level:
            node = list(self.root_hash.values())[0]

        # 2: Traverse down the tree
        while not node.is_terminal:
            node = node.route_to_child(query)

        # 3: Final node holds approximately similar items
        return node

    def exact_search(self, query):
        # 1: Get a best so far similarity through approximate search
        bsf_node = self.approximate_search(query)
        bsf_ts, bsf_dist = bsf_node.calc_min_dist(query)

        count = 0
        priority_queue = []

        # 2: Insert first nodes for evaluation
        for child in self.root_hash.values():
            child_lb = child.mindist_paa_isax(query)

            count += 1
            heapq.heappush(
                priority_queue,
                (child_lb, count, child)
            )

        # 3: Loop through candidates
        while priority_queue:
            current_lb, _, current_node = heapq.heappop(priority_queue)

            # Early stop: Best candidate has too high lower bound
            if current_lb >= bsf_dist:
                break
            
            if current_node.is_terminal:
                # 3.1: If node is leaf, check whether it holds more similar item
                current_ts, current_dist = current_node.calc_min_dist(query)

                if current_dist < bsf_dist:
                    bsf_dist = current_dist
                    bsf_ts = current_ts
            else:
                # 3.2: Traverse down the tree
                for child in current_node.children:
                    child_lb = child.mindist_paa_isax(query)
                  
                    count += 1
                    heapq.heappush(
                        priority_queue,
                        (child_lb, count, child)
                    )

        return bsf_ts, bsf_dist

## 3. Experiments

In [80]:
import numpy as np
import matplotlib.pyplot as plt

# ==========================================
# 1. Global Control Parameters
# ==========================================
n_ts = 1000           # Number of individual time series to generate
ts_length = 1000    # Number of data points per time series
noise_level = 1   # Standard deviation of the white noise

# Set seed for reproducibility (optional)
np.random.seed(42)

# ==========================================
# 2. Setup Dimensions for Broadcasting
# ==========================================
# Time vector reshaped to (ts_length, 1) to act as the vertical axis
t = np.linspace(0, 20, ts_length).reshape(-1, 1)

# ==========================================
# 3. Individual Random Characteristics
# ==========================================
# We generate arrays of shape (1, n_ts) so each series gets its own unique parameter

# Random frequencies for the sine and cosine waves (e.g., between 0.1 and 1.5 Hz)
freq_sin = np.random.uniform(0.1, 1.5, size=(1, n_ts))
freq_cos = np.random.uniform(0.1, 1.5, size=(1, n_ts))

# Random amplitudes (how tall the waves are)
amp_sin = np.random.uniform(2.0, 10.0, size=(1, n_ts))
amp_cos = np.random.uniform(2.0, 10.0, size=(1, n_ts))

# Random phase shifts (sliding the wave left or right)
phase_sin = np.random.uniform(0, 2 * np.pi, size=(1, n_ts))
phase_cos = np.random.uniform(0, 2 * np.pi, size=(1, n_ts))

# Random linear trends (drifting up or down over time)
trends = np.random.uniform(-3.0, 3.0, size=(1, n_ts))

# Random starting offsets (y-intercept)
intercepts = np.random.uniform(-20, 20, size=(1, n_ts))

# ==========================================
# 4. Construct the Time Series Set
# ==========================================
# Base Signals
wave_sin = amp_sin * np.sin(2 * np.pi * freq_sin * t + phase_sin)
wave_cos = amp_cos * np.cos(2 * np.pi * freq_cos * t + phase_cos)
linear_drift = (trends * t) + intercepts

# Noise Matrix of shape (ts_length, n_ts)
noise = np.random.normal(loc=0, scale=noise_level, size=(ts_length, n_ts))

# Combine all matrices! 
# ts_matrix is now a perfect (1000, 50) numpy array
ts_matrix = wave_sin + wave_cos + linear_drift + noise

# # ==========================================
# # 5. Visualization
# # ==========================================
# plt.figure(figsize=(14, 6))

# # Plot a subset (first 5 series) so the chart isn't an unreadable mess
# plt.plot(t, ts_matrix[:, :], alpha=0.85, linewidth=1.5)

# plt.title(f"Synthetic Dataset: 5 Random Samples (out of {n_ts} generated)", fontsize=14)
# plt.xlabel("Time", fontsize=12)
# plt.ylabel("Amplitude", fontsize=12)
# plt.grid(True, alpha=0.3)
# plt.show()

In [81]:
index = iSaxIndex(threshold=100, base_cardinality=2, word_length=8)

for ts in ts_matrix[1:, :]:
    index.insert(ts)

In [82]:
index.exact_search(ts_matrix[0, :])

(array([-2.69538746e+01, -2.40221838e+01,  7.63113128e+00,  6.62718637e+00,
         3.64451547e+00,  1.12666131e+01,  7.25866108e+00, -1.62718201e+01,
        -1.04915747e+01, -1.19390504e+01, -6.24630967e+00,  1.11476347e+01,
        -2.30881428e+01, -1.09411126e+01, -4.41384992e+00,  2.14611401e+01,
        -5.42678741e+00, -2.62787789e+01, -5.26045563e+00,  2.23424704e+01,
         4.05440771e+00,  1.42911775e+01,  1.76763536e+01,  1.02659032e+01,
         2.70360672e+00, -8.02436140e+00, -1.73892000e+01,  3.38503707e+01,
        -6.34014177e+00, -3.49991749e+00,  4.48289466e+00, -5.89682493e+00,
        -5.66165320e-01,  2.82659167e+00,  6.14198482e+00,  8.64311055e+00,
        -1.33471092e+01,  5.97006601e+00, -2.87866691e+00,  1.04277678e+01,
         6.49073376e+00, -6.34977479e+00,  7.36309046e+00, -1.94004082e+01,
         1.61278088e+01, -8.94856467e+00,  4.98845878e+00, -2.61717823e+01,
        -2.06781017e+01, -1.84812906e+00,  1.39352971e+01, -8.31284144e-01,
         1.6